In [6]:
# ==============================================================================
# 0. INSTALACE A IMPORTOVÁNÍ.....
# ==============================================================================
# Stáhne trénovací data a uloží je jako 'appartments_train.csv'
!wget -O appartments_train.csv https://raw.githubusercontent.com/LuciaKajanova/dspracticum25_flowers_team/refs/heads/main/homework/lesson_9/appartments_train.csv

# Stáhne testovací data a uloží je jako 'appartments_test.csv'
!wget -O appartments_test.csv https://raw.githubusercontent.com/LuciaKajanova/dspracticum25_flowers_team/refs/heads/main/homework/lesson_9/appartments_test.csv

print("\n Soubory jsou stažené a uložené v Colabu. Můžeš pokračovat dál!")
!pip install catboost xgboost lightgbm optuna scikit-learn pandas numpy scipy

import pandas as pd
import numpy as np
import re
import time
import optuna
from scipy.optimize import minimize
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, TargetEncoder, RobustScaler, QuantileTransformer
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import VotingRegressor, StackingRegressor, RandomForestRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.model_selection import train_test_split, KFold
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor, early_stopping
from catboost import CatBoostRegressor

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ==============================================================================
# ⚙️ 1. HLAVNÍ ŘÍDÍCÍ PANEL (MASTER CONFIG)
# ==============================================================================
USE_CLUSTERING = False          # Vypnuto na základě testů (ušetříme čas)
# ZDE PŘEPÍNEJ PRO SVŮJ VELKÝ TEST: "target" nebo "onehot"
ENCODING_METHOD = "target"      ## vypl jsem clustering i target encoding oba nepomahali

# --- NASTAVENÍ ---
RANDOM_STATE = 75
N_VALIDATION_RUNS = 10
N_TRIALS_OPTUNA = 18
N_FOLDS_OPTUNA = 5
OUTLIER_THRESHOLD = 0.003
##
threshold = OUTLIER_THRESHOLD
##
N_CLUSTERS = 50

# FIXNÍ PARAMETRY (Základ)
FIXED_PARAMS = {
    'n_estimators': 1000,       ### puvodne pak vratit 2500
    'early_stopping_rounds': 100,
    'n_jobs': -1,
    'random_state': RANDOM_STATE
}

# ==============================================================================
# 2. FEATURE ENGINEERING (VČETNĚ TIME FEATURES A PRAHY)
# ==============================================================================
#def remove_outliers(df, threshold):
    #df_c = df.copy()
    #for col in ['price', 'area']: df_c[col] = pd.to_numeric(df_c[col], errors='coerce')
    #df_c['price_m2'] = np.where((df_c['area'] > 1) & (df_c['price'] > 100000), df_c['price'] / df_c['area'], np.nan)
    #valid = df_c['price_m2'].dropna()
    #low, high = valid.quantile(threshold), valid.quantile(1 - threshold)
    #mask = ((df_c['price_m2'] >= low) & (df_c['price_m2'] <= high)) | df_c['price_m2'].isna()
    #return df_c[mask].drop(columns=['price_m2'])
def remove_outliers(df, threshold):
    df_c = df.copy()

    # 1. Převod na čísla (aby fungovalo porovnávání)
    for col in ['price', 'area', 'floor', 'total_floors']:
        if col in df_c.columns:
            df_c[col] = pd.to_numeric(df_c[col], errors='coerce')

    # ---------------------------------------------------------
    # NOVÁ OPRAVA: Patro vs. Celkem pater
    # ---------------------------------------------------------
    if 'floor' in df_c.columns and 'total_floors' in df_c.columns:
        # Kde je patro vyšší než celkový počet, nastavíme celkový počet = patro
        mask_bad_floors = df_c['floor'] > df_c['total_floors']
        df_c.loc[mask_bad_floors, 'total_floors'] = df_c.loc[mask_bad_floors, 'floor']
    # ---------------------------------------------------------

    # Pokračování původní logiky (bezpečné dělení atd.)
    df_c['price_m2'] = np.where((df_c.get('area', np.nan) > 0), df_c['price'] / df_c['area'], np.nan)

    if 'prague_district' not in df_c.columns:
        df_c['prague_district'] = 'All'

    def filter_group(x):
        low = x.quantile(threshold)
        high = x.quantile(1 - threshold)
        return x.between(low, high) | x.isna()

    mask = df_c.groupby('prague_district')['price_m2'].transform(filter_group)
    return df_c[mask].drop(columns=['price_m2', 'prague_district'], errors='ignore')

def get_prague_district(addr):
    addr = str(addr).lower()
    match = re.search(r'praha\s?-?(\d{1,2})', addr)
    if match: return f"Praha {match.group(1)}"
    return "Unknown"

def enhance_data(df, fit_kmeans=False, kmeans_model=None):
    df = df.copy()

    # --- 1. ČASOVÉ PŘÍZNAKY ---
    df['date_first'] = pd.to_datetime(df['first_seen'], errors='coerce')
    df['date_last'] = pd.to_datetime(df['last_seen'], errors='coerce')
    min_date = pd.Timestamp('2015-01-01')

    df['days_since'] = (df['date_first'] - min_date).dt.days.fillna(0)
    df['days_active'] = (df['date_last'] - df['date_first']).dt.days.fillna(0).clip(lower=0)

    # --- 2. PARSOVÁNÍ DISPOZIC ---
    def get_rooms(val):
        m = re.search(r'(\d+)', str(val))
        return int(m.group(1)) if m else 1
    df['n_rooms'] = df['layout'].apply(get_rooms)

    # Ošetření plochy a výpočet průměrné velikosti pokoje
    df['area'] = pd.to_numeric(df['area'], errors='coerce')
    df['avg_room_size'] = df['area'] / df['n_rooms'].replace(0, 1)

    # --- 3. SMART FILLING (Klíčové opravy) ---

    # A) Výtah: Chybí -> "Ne" (0)
    # Převedeme na binární, protože 'Unknown' nic neříká
    if 'elevator' in df.columns:
        df['elevator'] = df['elevator'].astype(str).str.lower().apply(
            lambda x: 1 if 'yes' in x or 'true' in x else 0
        )

    # B) Ostatní kategorické - zde je Unknown OK
    for col in ['ownership', 'condition', 'construction']:
        if col in df.columns: df[col] = df[col].fillna('Unknown')

    # C) Patra - logická imputace
    df['floor'] = pd.to_numeric(df.get('floor', 0), errors='coerce').fillna(0)
    df['total_floors'] = pd.to_numeric(df.get('total_floors', 0), errors='coerce')
    # Pokud nevíme total_floors, předpokládáme, že je to aspoň tolik, co floor (nebo 1)
    df['total_floors'] = df['total_floors'].fillna(df['floor']).replace(0, 1)

    df['relative_floor'] = df['floor'] / df['total_floors']
    df['is_top_floor'] = (df['floor'] >= df['total_floors']).astype(int)

    # D) POI Vzdálenosti - PENALIZACE (Místo mediánu)
    poi_dists = ['poi_transport_nearest', 'poi_grocery_nearest', 'poi_school_kindergarten_nearest',
                 'poi_doctors_nearest', 'poi_leisure_time_nearest', 'poi_restaurant_nearest']
    for col in poi_dists:
        if col in df.columns:
            val = pd.to_numeric(df[col], errors='coerce')
            # Pokud chybí, dáme 2x maximum (je to daleko)
            fill_val = val.max() * 2 if not pd.isna(val.max()) else 5000
            df[col] = np.log1p(val.fillna(fill_val))

    # E) POI Počty a Plochy - Zde nula dává smysl
    cols_to_zero = ['poi_transport_count', 'poi_grocery_count', 'poi_school_kindergarten_count',
                    'poi_doctors_count', 'poi_leisure_time_count', 'poi_restaurant_count',
                    'cellar_area', 'balcony_area', 'garden_area', 'parking']
    for col in cols_to_zero:
        if col in df.columns: df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    # --- 4. NOVÝ FEATURE: NET AREA ---
    df['net_area'] = df['area'] - df['balcony_area'] - df['cellar_area']
    # Pojistka proti záporným hodnotám
    df['net_area'] = df['net_area'].clip(lower=df['area'] * 0.5)

    # --- 5. LOKALITA ---
    def get_district(addr): return addr.split('-')[-1].strip() if isinstance(addr, str) and '-' in addr else "Unknown"
    df['district'] = df['address'].apply(get_district)
    df['prague_district'] = df['address'].apply(get_prague_district)

    df['gps_lat'] = pd.to_numeric(df['gps_lat'], errors='coerce')
    df['gps_lon'] = pd.to_numeric(df['gps_lon'], errors='coerce')
    df['dist_center'] = np.sqrt((df['gps_lat'] - 50.079)**2 + (df['gps_lon'] - 14.430)**2)

    # Vzdálenost k hradu
    hrad_lat, hrad_lon = 50.090, 14.400
    df['dist_hrad'] = np.sqrt((df['gps_lat'] - hrad_lat)**2 + (df['gps_lon'] - hrad_lon)**2)

    # Proximity Score
    df['proximity_score'] = 1 / (df['dist_center'] + 0.1)

    # --- 6. TEXTOVÉ FEATURES ---
    df['text'] = df['text'].astype(str).fillna('')
    df['text_lower'] = df['text'].str.lower()

    keywords = {
        'txt_metro': ['metro', 'metra', 'metru'],
        'txt_park': ['park', 'stromovk', 'letn', 'riegr', 'vítkov', 'vitkov'],
        'txt_water': ['vltav', 'nábřež', 'výhled na vodu', 'vyhled na vodu'],
        'txt_reconstructed': ['rekonstru', 'po oprav', 'nové jádro', 'zděné jádro', 'zdařil', 'zdaril'],
        'txt_new': ['novostav', 'nový byt', 'projekt', 'kolaudac', 'developersk'],
        'txt_brick': ['cihl', 'činžovní', 'činžovn', 'skelet'],
        'txt_balcony': ['balk', 'lodž', 'lodz', 'teras', 'zahrádk', 'předzahr'],
        'txt_parking': ['parkov', 'garáž', 'garaz', 'stání'],
        'txt_lift': ['výtah', 'vytah'],
        'txt_ac': ['klimatiza', 'klima'],
        'txt_storage': ['komor', 'šatn', 'sklep'],
        'txt_luxury': ['luxus', 'nadstandard', 'design', 'reziden'],
        'txt_view': ['výhled', 'vyhled', 'panoram', 'světlý', 'slunný'],
        'txt_high_ceilings': ['vysoké stropy', 'vysokými stropy'],
        'txt_bad_condition': ['původ', 'umakart', 'před rekonstru', 'k opravě'],
        'txt_low_floor': ['přízemí', 'prizemi', 'suterén'],
        'txt_coop': ['družst', 'druzst', 'družstevní']
    }
    for col, terms in keywords.items():
        pattern = '|'.join(terms)
        df[col] = df['text_lower'].str.contains(pattern, regex=True).astype(int)

    # Smart Fill (Categories based on text)
    #if 'ownership' in df.columns:
        #df.loc[df['ownership'] == 'Unknown', 'ownership'] = np.where(df.loc[df['ownership'] == 'Unknown', 'txt_coop'] == 1, 'Družstevní', 'Unknown')

    #if 'construction' in df.columns:
        #df.loc[df['construction'] == 'Unknown', 'construction'] = np.where(df.loc[df['construction'] == 'Unknown', 'txt_brick'] == 1, 'Cihlová', 'Unknown')

    #if 'txt_luxury' in df.columns: df['luxury_size'] = df['area'] * df['txt_luxury']

    # --- 7. MICRO-SEGMENTACE ---
    df['segment'] = df['prague_district'].astype(str) + '_' + df['construction'].astype(str)

    # --- 8. CLUSTERING (Volitelné) ---
    coords = df[['gps_lat', 'gps_lon']].fillna(0)
    if USE_CLUSTERING and fit_kmeans:
        kmeans_model = KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=RANDOM_STATE)
        df['geo_cluster'] = kmeans_model.fit_predict(coords).astype(str)
    elif USE_CLUSTERING and kmeans_model:
        df['geo_cluster'] = kmeans_model.predict(coords).astype(str)
    else:
        df['geo_cluster'] = "0"

    df = df.drop(columns=['text_lower', 'date_first', 'date_last'], errors='ignore')
    return df, kmeans_model

    # ... (tvé existující kódy v enhance_data) ...################################################

    # 1. GAMECHANGER: MICRO-SEGMENTACE (NECHAT - TO JE TEN NEJSILNĚJŠÍ)
    # Spojíme Prahu a Konstrukci. Model uvidí rozdíl mezi "Praha 4_Panel" a "Praha 4_Cihla".
    df['segment'] = df['prague_district'].astype(str) + '_' + df['construction'].astype(str)

    # 2. GAMECHANGER: VLASTNICTVÍ (VYPUSTIT PRO RYCHLOST)
    #df['segment_own'] = ... (Tento řádek jsme smazali)

    # 3. GAMECHANGER: INVERZNÍ VZDÁLENOST (NECHAT - JE TO ZADARMO)
    # Pomůže modelu pochopit, že cena v centru roste exponenciálně.
    df['gps_lat'] = pd.to_numeric(df['gps_lat'], errors='coerce')
    df['gps_lon'] = pd.to_numeric(df['gps_lon'], errors='coerce')

    # 2. PRAŽSKÝ HRAD (Historické/Turistické centrum)
    # Souřadnice: Katedrála sv. Víta
    hrad_lat, hrad_lon = 50.090, 14.400
    df['dist_hrad'] = np.sqrt((df['gps_lat'] - hrad_lat)**2 + (df['gps_lon'] - hrad_lon)**2)

    # 3. GAMECHANGER: PROXIMITY SCORE (Vylepšené)
    # Modelu říkáme: "Být 500m od Václaváku je 10x lepší než být 5km daleko"
    # Kombinujeme oba středy (vybere si ten bližší)
    #df['min_dist_center'] = df[['dist_center', 'dist_hrad']].min(axis=1)
    df['proximity_score'] = 1 / (df['dist_center'] + 0.1)

    # Do enhance_data:
    # Vltava teče cca po poledníku 14.41 (zjednodušeně)
    #df['dist_river'] = np.abs(df['gps_lon'] - 14.41)

    ############################################################################################

    df = df.drop(columns=['text_lower', 'date_first', 'date_last'], errors='ignore')
    return df, kmeans_model


--2025-11-30 14:21:57--  https://raw.githubusercontent.com/LuciaKajanova/dspracticum25_flowers_team/refs/heads/main/homework/lesson_9/appartments_train.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9033771 (8.6M) [text/plain]
Saving to: ‘appartments_train.csv’

appartments_train.c 100%[===================>]   8.62M  --.-KB/s    in 0.06s   

2025-11-30 14:21:57 (143 MB/s) - ‘appartments_train.csv’ saved [9033771/9033771]

--2025-11-30 14:21:57--  https://raw.githubusercontent.com/LuciaKajanova/dspracticum25_flowers_team/refs/heads/main/homework/lesson_9/appartments_test.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubuserconte

priprava dat

In [7]:

# ==============================================================================
# 3. PŘÍPRAVA DAT (TF-IDF + TIME)
# ==============================================================================
try: train_raw = pd.read_csv("appartments_train.csv"); test = pd.read_csv("appartments_test.csv")
except: train_raw = pd.read_csv("apartments_train.csv"); test = pd.read_csv("apartments_test.csv")
print(f" Data načtena: Train {train_raw.shape}")

train_c = remove_outliers(train_raw, OUTLIER_THRESHOLD)
y_all = np.log1p(train_c["price"])
X_all_e, km_model = enhance_data(train_c, fit_kmeans=True)

#  DŮLEŽITÉ: 'text' NEVYHAZUJEME, ABY HO MOHL VZÍT TF-IDF
drop_cols = ["price", "id", "address", "text", "first_seen", "last_seen", "layout", "price_m2", "date_ts"]
X_all = X_all_e.drop(columns=drop_cols, errors='ignore')

# --- PREPROCESSOR ---
num_cols = X_all.select_dtypes(include=[np.number]).columns.tolist()
cat_all = X_all.select_dtypes(exclude=[np.number]).columns.tolist()
# Text vyjmeme z cat_all, aby šel do TF-IDF
if 'text' in cat_all: cat_all.remove('text')

cat_target_candidates = ['district', 'prague_district']
if USE_CLUSTERING: cat_target_candidates.append('geo_cluster')
cat_onehot_candidates = [c for c in cat_all if c not in cat_target_candidates]

if ENCODING_METHOD == "target":
    cat_transformers = [
        ("target_enc", TargetEncoder(target_type="continuous", smooth=10.0, cv=5), cat_target_candidates),
        ("onehot_enc", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="Unknown")), ("enc", OneHotEncoder(handle_unknown="ignore", sparse_output=False, min_frequency=2))]), cat_onehot_candidates)
    ]
else:
    cat_all_for_ohe = cat_target_candidates + cat_onehot_candidates
    cat_transformers = [
        ("onehot_enc", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="Unknown")), ("enc", OneHotEncoder(handle_unknown="ignore", sparse_output=False, min_frequency=2))]), cat_all_for_ohe)
    ]

prep = ColumnTransformer([
    ("area_fix", SimpleImputer(strategy="constant", fill_value=0), [c for c in num_cols if 'area' in c]),
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("scaler", RobustScaler()), ("quant", QuantileTransformer(output_distribution='normal', n_quantiles=500))]), [c for c in num_cols if 'area' not in c]),
] + cat_transformers)
#  Smazal jsem ten řádek: + [ ("tfidf", ... text) ]

print(f" Preprocessor: {ENCODING_METHOD.upper()} | Time Features Active")

X_preprocessed = prep.fit_transform(X_all, y_all)
y_target = y_all


 Data načtena: Train (5000, 32)
 Preprocessor: TARGET | Time Features Active


optuna


In [8]:

# ----------------------------------------------------------------------
# 5) OPTUNA - objective functions (používají X_preprocessed)
# ----------------------------------------------------------------------
def objective_xgb(trial):
    params = {
        'n_estimators': FIXED_PARAMS['n_estimators'],
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 7.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 7.0, log=True),
        'n_jobs': -1, 'random_state': RANDOM_STATE, 'tree_method': 'hist'
    }
    model = XGBRegressor(**params, early_stopping_rounds=FIXED_PARAMS['early_stopping_rounds'])
    kf = KFold(n_splits=N_FOLDS_OPTUNA, shuffle=True, random_state=RANDOM_STATE)
    scores = []
    for tr_idx, va_idx in kf.split(X_preprocessed):
        model.fit(X_preprocessed[tr_idx], y_target.iloc[tr_idx],
                  eval_set=[(X_preprocessed[va_idx], y_target.iloc[va_idx])], verbose=False)
        scores.append(mean_absolute_percentage_error(np.expm1(y_target.iloc[va_idx]), np.expm1(model.predict(X_preprocessed[va_idx]))) * 100)
    return float(np.mean(scores))


def objective_lgbm(trial):
    params = {
        'n_estimators': FIXED_PARAMS['n_estimators'],
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 16, 128),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 7.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 7.0, log=True),
        'n_jobs': -1, 'random_state': RANDOM_STATE, 'verbose': -1
    }
    kf = KFold(n_splits=N_FOLDS_OPTUNA, shuffle=True, random_state=RANDOM_STATE)
    scores = []
    for tr_idx, va_idx in kf.split(X_preprocessed):
        model = LGBMRegressor(**params)
        model.fit(X_preprocessed[tr_idx], y_target.iloc[tr_idx],
                  eval_set=[(X_preprocessed[va_idx], y_target.iloc[va_idx])],
                  callbacks=[early_stopping(FIXED_PARAMS['early_stopping_rounds'], verbose=False)])
        scores.append(mean_absolute_percentage_error(np.expm1(y_target.iloc[va_idx]), np.expm1(model.predict(X_preprocessed[va_idx]))) * 100)
    return float(np.mean(scores))


def objective_cat(trial):
    params = {
        'iterations': FIXED_PARAMS['n_estimators'],
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 8),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 7.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'loss_function': 'MAE', 'random_seed': RANDOM_STATE, 'verbose': 0, 'allow_writing_files': False
    }
    kf = KFold(n_splits=N_FOLDS_OPTUNA, shuffle=True, random_state=RANDOM_STATE)
    scores = []
    for tr_idx, va_idx in kf.split(X_preprocessed):
        model = CatBoostRegressor(**params)
        model.fit(X_preprocessed[tr_idx], y_target.iloc[tr_idx],
                  eval_set=(X_preprocessed[va_idx], y_target.iloc[va_idx]),
                  early_stopping_rounds=FIXED_PARAMS['early_stopping_rounds'], verbose=False)
        scores.append(mean_absolute_percentage_error(np.expm1(y_target.iloc[va_idx]), np.expm1(model.predict(X_preprocessed[va_idx]))) * 100)
    return float(np.mean(scores))


def objective_rf(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 500),
        'max_depth': trial.suggest_int('max_depth', 10, 30),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 4),
        'max_features': trial.suggest_float('max_features', 0.5, 1.0),
        'n_jobs': -1, 'random_state': RANDOM_STATE
    }
    kf = KFold(n_splits=N_FOLDS_OPTUNA, shuffle=True, random_state=RANDOM_STATE)
    scores = []
    for tr_idx, va_idx in kf.split(X_preprocessed):
        model = RandomForestRegressor(**params)
        model.fit(X_preprocessed[tr_idx], y_target.iloc[tr_idx])
        scores.append(mean_absolute_percentage_error(np.expm1(y_target.iloc[va_idx]), np.expm1(model.predict(X_preprocessed[va_idx]))) * 100)
    return float(np.mean(scores))


# spustíme Optunu (ticha)
sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
print(f"\n1. FÁZE: OPTUNA ({N_TRIALS_OPTUNA} trials)...")

study_xgb = optuna.create_study(direction='minimize', sampler=sampler)
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS_OPTUNA)
best_xgb_params = study_xgb.best_params
print(f"XGB Best: {study_xgb.best_value:.3f}%")

study_lgbm = optuna.create_study(direction='minimize', sampler=sampler)
study_lgbm.optimize(objective_lgbm, n_trials=N_TRIALS_OPTUNA)
best_lgbm_params = study_lgbm.best_params
print(f"LGBM Best: {study_lgbm.best_value:.3f}%")

study_cat = optuna.create_study(direction='minimize', sampler=sampler)
study_cat.optimize(objective_cat, n_trials=N_TRIALS_OPTUNA)
best_cat_params = study_cat.best_params
print(f"CAT Best: {study_cat.best_value:.3f}%")

study_rf = optuna.create_study(direction='minimize', sampler=sampler)
study_rf.optimize(objective_rf, n_trials=N_TRIALS_OPTUNA)
best_rf_params = study_rf.best_params
print(f"RF Best: {study_rf.best_value:.3f}%")


# Update Params
best_xgb_params.update({'n_estimators': 6000, 'n_jobs': -1, 'random_state': RANDOM_STATE, 'tree_method': 'hist'})
best_lgbm_params.update({'n_estimators': 6000, 'n_jobs': -1, 'random_state': RANDOM_STATE, 'verbose': -1, 'early_stopping_rounds': None})
best_cat_params.update({'iterations': 6000, 'loss_function': 'MAE', 'random_seed': RANDOM_STATE, 'verbose': 0, 'allow_writing_files': False, 'early_stopping_rounds': None})
best_rf_params.update({'n_jobs': -1, 'random_state': RANDOM_STATE})



1. FÁZE: OPTUNA (18 trials)...
XGB Best: 9.054%


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

LGBM Best: 9.118%
CAT Best: 9.187%
RF Best: 10.140%


validace

In [9]:

# ==============================================================================
# 6. FÁZE 2: VALIDACE (SOLVER VAH + UPDATED DROP_COLS)
# ==============================================================================
print(f"\n{'='*85}\n 2. FÁZE: VALIDACE ({N_VALIDATION_RUNS} runs) s OPTIMALIZACÍ VAH\n{'='*85}")

res_voting = []
res_stacking = []
optimal_iters = {"xgb": [], "lgbm": [], "cat": []}
collected_weights = []

for run_i in range(1, N_VALIDATION_RUNS + 1):
    seed = RANDOM_STATE + (run_i ) * 7

    train_d, val_d = train_test_split(train_raw, test_size=0.2, random_state=seed)
    train_c = remove_outliers(train_d, OUTLIER_THRESHOLD)
    y_train = np.log1p(train_c["price"])
    y_val = np.log1p(val_d["price"])

    X_tr, km = enhance_data(train_c, fit_kmeans=True)
    X_va, _ = enhance_data(val_d, kmeans_model=km)

    # Text nemažeme, ale ošetříme
    drop_cols_loop = ["price", "id" , "address","text" ,"first_seen", "last_seen", "layout", "price_m2", "date_ts"]
    X_tr_fin = X_tr.drop(columns=drop_cols_loop, errors='ignore')
    X_va_fin = X_va.drop(columns=drop_cols_loop, errors='ignore')

    X_tr_ready = prep.fit_transform(X_tr_fin, y_train)
    X_va_ready = prep.transform(X_va_fin)

    print(f"Run {run_i}: Training...", end="")

    # 1. XGB (Opravený)
    best_xgb_params['early_stopping_rounds'] = 300
    xgb = XGBRegressor(**best_xgb_params)
    xgb.fit(X_tr_ready, y_train, eval_set=[(X_va_ready, y_val)], verbose=False)
    try: optimal_iters["xgb"].append(xgb.best_iteration)
    except: optimal_iters["xgb"].append(xgb.n_estimators)

    lgbm = LGBMRegressor(**best_lgbm_params)
    lgbm.fit(X_tr_ready, y_train, eval_set=[(X_va_ready, y_val)], callbacks=[early_stopping(300, verbose=False)])
    optimal_iters["lgbm"].append(lgbm.best_iteration_)

    cat = CatBoostRegressor(**best_cat_params)
    cat.fit(X_tr_ready, y_train, eval_set=(X_va_ready, y_val), early_stopping_rounds=300, verbose=False)
    optimal_iters["cat"].append(cat.best_iteration_)

    rf = RandomForestRegressor(**best_rf_params)
    rf.fit(X_tr_ready, y_train)

    print(" Done.")

    p_x = xgb.predict(X_va_ready)
    p_l = lgbm.predict(X_va_ready)
    p_c = cat.predict(X_va_ready)
    p_r = rf.predict(X_va_ready)

    # --- SOLVER VAH ---
    preds_matrix = np.column_stack([np.expm1(p_x), np.expm1(p_l), np.expm1(p_c), np.expm1(p_r)])
    true_prices = np.expm1(y_val)

    def mape_loss(weights):
        final_p = np.average(preds_matrix, axis=1, weights=weights)
        return mean_absolute_percentage_error(true_prices, final_p)

    init_w = [0.25, 0.25, 0.25, 0.25]
    bounds = [(0, 1)] * 4
    cons = ({'type': 'eq', 'fun': lambda w: 1 - sum(w)})

    res = minimize(mape_loss, init_w, method='SLSQP', bounds=bounds, constraints=cons)
    opt_w = res.x
    collected_weights.append(opt_w)

    print(f"    Opt. Váhy: XGB:{opt_w[0]:.2f}, LGBM:{opt_w[1]:.2f}, CAT:{opt_w[2]:.2f}, RF:{opt_w[3]:.2f}")

    y_pred_vote = np.average(preds_matrix, axis=1, weights=opt_w)
    mape_v = mean_absolute_percentage_error(true_prices, y_pred_vote) * 100
    res_voting.append(mape_v)

    meta_X = np.column_stack([p_x, p_l, p_c, p_r])
    meta_model = RidgeCV(alphas=[0.1, 1.0, 10.0])
    meta_model.fit(meta_X, y_val)
    y_pred_stack = meta_model.predict(meta_X)
    mape_s = mean_absolute_percentage_error(np.expm1(y_val), np.expm1(y_pred_stack)) * 100
    res_stacking.append(mape_s)

    print(f"   VOTING (Opt): {mape_v:.3f}% | STACKING: {mape_s:.3f}%")
    last_y_val, last_y_pred_stack = y_val, y_pred_stack

# ==============================================================================
# 7. FÁZE 3: FINÁLNÍ TRÉNINK NA 100% DAT
# ==============================================================================
print("\n 3. FÁZE: GENERUJI FINÁLNÍ SUBMISSIONS...")

train_full = remove_outliers(train_raw, OUTLIER_THRESHOLD)
y_full = np.log1p(train_full["price"])

X_full_e, km_full = enhance_data(train_full, fit_kmeans=True)
X_test_e, _ = enhance_data(test, kmeans_model=km_full)

X_full_fin = X_full_e.drop(columns=drop_cols_loop, errors='ignore')
X_test_fin = X_test_e.drop(columns=drop_cols_loop, errors='ignore')

X_full_ready = prep.fit_transform(X_full_fin, y_full)
X_test_ready = prep.transform(X_test_fin)

ix = int(np.mean(optimal_iters["xgb"]) * 1.1)
il = int(np.mean(optimal_iters["lgbm"]) * 1.1)
ic = int(np.mean(optimal_iters["cat"]) * 1.1)

best_xgb_params.update({'n_estimators': ix, 'early_stopping_rounds': None})
best_lgbm_params.update({'n_estimators': il, 'early_stopping_rounds': None})
best_cat_params.update({'iterations': ic, 'early_stopping_rounds': None})

print(f"   Trénuji finální modely (XGB:{ix}, LGBM:{il}, CAT:{ic} iters)...")
xgb_f = XGBRegressor(**best_xgb_params); xgb_f.fit(X_full_ready, y_full)
lgbm_f = LGBMRegressor(**best_lgbm_params); lgbm_f.fit(X_full_ready, y_full)
cat_f = CatBoostRegressor(**best_cat_params); cat_f.fit(X_full_ready, y_full)
rf_f = RandomForestRegressor(**best_rf_params); rf_f.fit(X_full_ready, y_full)

p_x = xgb_f.predict(X_test_ready); p_l = lgbm_f.predict(X_test_ready); p_c = cat_f.predict(X_test_ready); p_r = rf_f.predict(X_test_ready)

# ... (kód po trénink modelů xgb_f, lgbm_f atd.) ...

# 1. Voting Export
final_weights = np.mean(collected_weights, axis=0)
# Normalizace vah pro jistotu, aby součet byl 1
final_weights /= np.sum(final_weights)

preds_matrix_test_real = np.column_stack([np.expm1(p_x), np.expm1(p_l), np.expm1(p_c), np.expm1(p_r)])
final_vote = np.average(preds_matrix_test_real, axis=1, weights=final_weights)

# Ošetření záporných cen (pro jistotu) a zaokrouhlení
final_vote = np.maximum(0, final_vote)
pd.DataFrame({"id": test["id"], "price": final_vote.round(0).astype(int)}).to_csv("submission_OPTUNA_VOTING.csv", index=False)

# 2. Stacking Export
print("    Trénuji Stacking Regressor...")
ests = [('xgb', xgb_f), ('lgbm', lgbm_f), ('cat', cat_f), ('rf', rf_f)]
# Zvýšení počtu splitů pro Stacking, aby meta-model měl více dat
cv_stack = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

stack = StackingRegressor(estimators=ests, final_estimator=RidgeCV(), n_jobs=-1, cv=cv_stack)
stack.fit(X_full_ready, y_full)
final_stack = np.expm1(stack.predict(X_test_ready))

# Vypnutí riskantní Bias Correction (pokud nemáš spočítaný bias přes všechna data)
# bias_factor = ... (zakomentováno)
final_stack_corrected = np.maximum(0, final_stack) # Jen pojistka proti < 0

pd.DataFrame({"id": test["id"], "price": final_stack_corrected.round(0).astype(int)}).to_csv("submission_OPTUNA_STACKING.csv", index=False)



 2. FÁZE: VALIDACE (10 runs) s OPTIMALIZACÍ VAH
Run 1: Training... Done.


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


    Opt. Váhy: XGB:0.40, LGBM:0.33, CAT:0.27, RF:0.00
   VOTING (Opt): 9.303% | STACKING: 9.341%
Run 2: Training... Done.


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


    Opt. Váhy: XGB:0.50, LGBM:0.31, CAT:0.20, RF:0.00
   VOTING (Opt): 9.475% | STACKING: 9.437%
Run 3: Training... Done.


...

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


    Opt. Váhy: XGB:0.36, LGBM:0.31, CAT:0.33, RF:0.00
   VOTING (Opt): 8.666% | STACKING: 8.708%

 3. FÁZE: GENERUJI FINÁLNÍ SUBMISSIONS...
   Trénuji finální modely (XGB:1588, LGBM:1779, CAT:3834 iters)...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


    Trénuji Stacking Regressor...


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


report

In [10]:
# ==============================================================================
# 8. FINÁLNÍ REPORT
# ==============================================================================
print(f"\n{'='*85}\n FINÁLNÍ REPORT \n{'='*85}")

# --- 1. SOUHRNNÉ VÝSLEDKY ---
print(f"| Strategie | Ø MAPE | Std. Dev. |")
print(f"| :--- | :--- | :--- |")
print(f"| VOTING (Opt) | {np.mean(res_voting):.3f}% | {np.std(res_voting):.3f} |")
print(f"| STACKING | {np.mean(res_stacking):.3f}% | {np.std(res_stacking):.3f} |")

# --- 2. VÝPIS VÍTĚZNÝCH HYPERPARAMETRŮ (Diagnostika modelu) ---
print(f"\n{'='*85}")
print(" OPTUNA DIAGNOSTIKA: VÍTĚZNÉ PARAMETRY (PRO STABILITU)")
print(f"{'='*85}")
try:
    # 1. XGBOOST
    print(f"\n XGBoost Best MAPE: {study_xgb.best_value:.4f}%")
    print("\n--- XGBoost Parametry ---")
    xgb_params_df = pd.Series(study_xgb.best_params, name="XGBoost Hodnoty")
    print(xgb_params_df.to_markdown())

    # 2. LIGHTGBM
    print("\n--- LightGBM Parametry ---")
    lgbm_params_df = pd.Series(study_lgbm.best_params, name="LightGBM Hodnoty")
    print(lgbm_params_df.to_markdown())

    # 3. CATBOOST
    print("\n--- CatBoost Parametry ---")
    cat_params_df = pd.Series(study_cat.best_params, name="CatBoost Hodnoty")
    print(cat_params_df.to_markdown())

except Exception as e:
    print(f" CHYBA: Nepodařilo se vypsat Optuna parametry. Prosím, zkontroluj Fázi 1.")


# --- 4. DETAILNÍ VÝSLEDKY VALIDACE (Run by Run) ---
print("\n## 4. DETAILNÍ VÝSLEDKY VALIDACE (Run by Run)")
validation_results = []
for i in range(N_VALIDATION_RUNS):
    validation_results.append({
        'Run': i + 1,
        'STACKING (%)': res_stacking[i],
        'VOTING (%)': res_voting[i],
        'XGB_W': collected_weights[i][0],
        'LGBM_W': collected_weights[i][1],
        'CAT_W': collected_weights[i][2],
        'RF_W': collected_weights[i][3] if len(collected_weights[i]) > 3 else 0.0,
    })

val_df = pd.DataFrame(validation_results)
print(val_df.to_markdown(index=False, floatfmt=".3f"))

# ==============================================================================
# 9. ANALÝZA FEATURE IMPORTANCE A KONTROLA VÝSTUPU
# ==============================================================================
print(f"\n{'='*85}")
print(" DETAILNÍ ANALÝZA: CO ROZHODUJE O CENĚ (XGBoost)")
print(f"{'='*85}")

feature_names = []
try:
    for name, transformer, features in prep.transformers_:
        if name == 'num' or name == 'area_fix':
            if hasattr(transformer, 'get_feature_names_out'):
                 feature_names.extend(transformer.get_feature_names_out(features))
            else:
                 feature_names.extend(features)
        elif name == 'onehot_enc':
            try:
                ohe = transformer.named_steps['enc']
                feature_names.extend(ohe.get_feature_names_out(features))
            except:
                feature_names.extend(features)
        elif name == 'target_enc':
             feature_names.extend(features)
        elif name == 'tfidf':
            feature_names.extend([f"txt_{word}" for word in transformer.get_feature_names_out()])
        else:
            pass

    if len(feature_names) != len(xgb_f.feature_importances_):
        print(f" Pozor: Počet jmen ({len(feature_names)}) nesedí s počtem features v modelu ({len(xgb_f.feature_importances_)}).")
        print("Zobrazuji jen raw importances bez názvů (pro jistotu).")
        imp_df = pd.DataFrame({'Feature': [f"Feature_{i}" for i in range(len(xgb_f.feature_importances_))],
                               'Importance': xgb_f.feature_importances_})
    else:
        imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': xgb_f.feature_importances_})

    print(imp_df.sort_values(by='Importance', ascending=False).head(40).to_markdown(index=False, floatfmt=".4f"))

except Exception as e:
    print(f" Nepodařilo se vykreslit Feature Importance: {e}")
    print("Tip: Ujisti se, že jsi spustil celou Fázi 3 a model 'xgb_f' existuje.")

# --- FINÁLNÍ KONTROLA FORMÁTU A CENY (Nově zaokrouhleno) ---
try:
    submission_df = pd.read_csv("submission_OPTUNA_STACKING.csv")

    #  OPRAVA VÝSTUPU: Zaokrouhlíme cenu na celé číslo, abychom se zbavili notace e+06
    submission_df['price'] = submission_df['price'].round(0).astype(int)

    print("\n Kontrola finálního formátu (cena v Kč):")
    print(submission_df.head(5).to_markdown(index=False))
    print(f"\nCelkový počet predikcí: {len(submission_df)}")

except Exception as e:
    print(f" CHYBA PŘI ZOBRAZENÍ VÝSTUPU: {e}")




 FINÁLNÍ REPORT 
| Strategie | Ø MAPE | Std. Dev. |
| :--- | :--- | :--- |
| VOTING (Opt) | 9.392% | 0.477 |
| STACKING | 9.364% | 0.442 |

 OPTUNA DIAGNOSTIKA: VÍTĚZNÉ PARAMETRY (PRO STABILITU)

 XGBoost Best MAPE: 9.0541%

--- XGBoost Parametry ---
|                  |   XGBoost Hodnoty |
|:-----------------|------------------:|
| learning_rate    |         0.024049  |
| max_depth        |         6         |
| subsample        |         0.515716  |
| colsample_bytree |         0.771487  |
| reg_alpha        |         0.13957   |
| reg_lambda       |         0.0381562 |

--- LightGBM Parametry ---
|                  |   LightGBM Hodnoty |
|:-----------------|-------------------:|
| learning_rate    |         0.0268775  |
| num_leaves       |        52          |
| subsample        |         0.808992   |
| colsample_bytree |         0.572209   |
| reg_alpha        |         0.0345117  |
| reg_lambda       |         0.00107968 |

--- CatBoost Parametry ---
|               |   CatBoost